In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product

import time
import sys
import requests
import logging
import os

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from scipy import stats
from scipy.optimize import minimize
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from statsmodels.tsa.api import SimpleExpSmoothing, Holt, ExponentialSmoothing

In [2]:
df = pd.read_excel("pmovdcE CNH 2 Jul 26.xlsx", skiprows=4, sheet_name="CNH Non Steron")

In [3]:
import logging
#MULTIPLE AGC df_all ganti ke df_final nanti
logging.info("BEGIN Constructing All Branch Data and Combining It to DF")

# Normalize columns
df.columns = df.columns.str.lower()
df["p/n"] = df["p/n"].str.upper()

# Demand columns sorted from D-16 to D-1
demand_columns = sorted(
    [col for col in df.columns if col.startswith("d-")],
    key=lambda x: int(x.split("-")[1]),
    reverse=True
)

# ---------- Step 1: Convert demand columns to list for each row ----------
df["d"] = df[demand_columns].values.tolist()
df = df[["brc", "agc", "p/n", "d"]]

# ---------- Step 2: Aggregate to create full National summary (ALL AGC) ----------
df_national_all_agc = df.groupby(["p/n"], as_index=False).agg({
    "d": lambda rows: [sum(x) for x in zip(*rows)]
})
df_national_all_agc.insert(0, "brc", "National")
df_national_all_agc.insert(1, "agc", "ALL AGC")

# ---------- Step 3: Combine everything ----------
df_all = pd.concat([df, df_national_all_agc], ignore_index=True)

# ---------- Step 4: Drop rows with brc == 'National' and agc != 'ALL AGC' ----------
df_all = df_all[~((df_all["brc"] == "National") & (df_all["agc"] != "ALL AGC"))]

logging.info(f"Constructed DF with Per Branch and National ALL AGC only — Total: {len(df_all)} rows")
print(df_all)


            brc      agc            p/n  \
0            20        6       47364317   
1            20        6       47486870   
2            20        6       48017998   
3            20        6       48167835   
4            20        6       48174869   
...         ...      ...            ...   
87146  National  ALL AGC   VI8973679180   
87147  National  ALL AGC  VV12961252100   
87148  National  ALL AGC       X1139855   
87149  National  ALL AGC      XF0202MDN   
87150  National  ALL AGC       Z2230493   

                                                      d  
0      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  
1      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  
2      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  
3      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  
4      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  
...                                                 ...  
87146  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  
87147  [0, 0, 0, 0,

In [4]:
# Calculate Forecast
logging.info("BEGIN Forecast Calculation")
# display(df)

In [5]:
logging.info("BEGIN Mean, Std, UB Calculation, and Construct Clipping Data")

# Get mean and standard deviation of 12 periods before the last one
df_all["d"] = df_all["d"].apply(lambda x: x if isinstance(x, list) else [])  # Ensure d is a list
df_all['mean_12'] = df_all['d'].apply(lambda x: np.mean(x[-13:-1]))  # Use 12 periods before the last one
df_all['std_12'] = df_all['d'].apply(lambda x: np.std(x[-13:-1]))    # Use 12 periods before the last one

# Get upper bound from mean and std
df_all['ub'] = df_all['mean_12'] + 1.5 * df_all['std_12']

# Limit the original df to upper bound (using the 12 periods before the last one)
df_all['clipped_d'] = df_all.apply(lambda row: np.clip(row['d'][-13:-1], 0, row['ub']).tolist(), axis=1)

# Display the updated DataFrame
display(df_all)


,brc,agc,p/n,d,mean_12,std_12,ub,clipped_d
0,20,6,47364317,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,20,6,47486870,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,20,6,48017998,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,20,6,48167835,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,20,6,48174869,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...,...,...,...,...,...
87146,National,ALL AGC,VI8973679180,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
87147,National,ALL AGC,VV12961252100,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
87148,National,ALL AGC,X1139855,"[0, 0, 0, 0, 0, 0, 1, 0, 3, 0, 0, 0, 0, 0, 0, 0]",0.333333,0.849837,1.608088,"[0.0, 0.0, 0.0, 1.0, 0.0, 1.6080882117315294, ..."
87149,National,ALL AGC,XF0202MDN,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [6]:
logging.info("BEGIN Moving Average Calculation")

# Calculate Simple Moving Average
df_all['clipped_d_15'] = df_all.apply(lambda row: np.clip(row['d'][:15], 0, row['ub']).tolist(), axis=1)

# Function to compute SMA forecasts for D-13 to D-1 using 3-point averages
def sma_forecast(data):
    sma_values = []
    for i in range(13):  # We want 13 forecast points: D-13 to D-1
        window = data[i:i+3]
        forecast = np.mean(window)  # Equal weights
        sma_values.append(forecast)
    return sma_values

# Apply SMA forecasting logic
df_all['ma'] = df_all['clipped_d_15'].apply(sma_forecast)

# Extract the last forecast (for D-1)
df_all['ma_result'] = df_all['ma'].apply(lambda x: x[-1])

print(df_all)

            brc      agc            p/n  \
0            20        6       47364317   
1            20        6       47486870   
2            20        6       48017998   
3            20        6       48167835   
4            20        6       48174869   
...         ...      ...            ...   
87146  National  ALL AGC   VI8973679180   
87147  National  ALL AGC  VV12961252100   
87148  National  ALL AGC       X1139855   
87149  National  ALL AGC      XF0202MDN   
87150  National  ALL AGC       Z2230493   

                                                      d   mean_12    std_12  \
0      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
1      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
2      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
3      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
4      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
...      

In [7]:
import numpy as np
import pandas as pd

logging.info("BEGIN Weighted Moving Average Calculation")


# Function to compute WMA forecasts for D-13 to D-1
def wma_forecast_with_weights(data, weights):
    wma_values = []
    for i in range(13):  # Forecasting D-13 to D-1 using D-16 to D-2
        window = data[i:i+3]
        forecast = np.sum(np.array(window) * weights) / sum(weights)
        wma_values.append(forecast)
    return wma_values

# Define step size
step = 0.05

# Initialize columns to store best weights, forecasts, and WMA results
df_all['wma_best_w1'] = np.nan
df_all['wma_best_w2'] = np.nan
df_all['wma_best_w3'] = np.nan
df_all['wma_result'] = np.nan
df_all['wma_forecast'] = df_all.apply(lambda _: [], axis=1)  # Initialize as empty lists

# Optimize weights for each row
for idx, row in df_all.iterrows():
    best_rmse = float('inf')
    best_weights = (0.15, 0.25, 0.6)  # Initial weight assumption
    best_forecast = None
    best_full_forecast = None  # Store full forecast array

    # Iterate over valid w1 values
    for w1 in np.round(np.arange(0.15, 0.81, step), 2):  # w1 ≥ 0.15
        for w2 in np.round(np.arange(0.25, 0.86 - w1, step), 2):  # w2 ≥ 0.25 and w1 + w2 ≤ 0.85
            w3 = 1 - (w1 + w2)  # Ensure sum is exactly 1

            # Ensure w3 > w2 > w1
            if w3 > w2 > w1:
                weights = (w1, w2, w3)

                # Compute WMA forecast for this row
                wma_forecast = wma_forecast_with_weights(row['clipped_d_15'], weights)

                # Extract the D-1 prediction (last forecast)
                wma_result = wma_forecast[-1]

                # Extract actual last value of 'd' (D-1)
                d_last = row['d'][-1]

                # Compute RMSE for this row
                rmse = np.sqrt((d_last - wma_result) ** 2)

                # Store best weights if RMSE improves
                if rmse < best_rmse:
                    best_rmse = rmse
                    best_weights = weights
                    best_forecast = wma_result
                    best_full_forecast = wma_forecast  # Store full forecast

    # Store the best weights and forecast for this row
    df_all.at[idx, 'wma_best_w1'] = best_weights[0]
    df_all.at[idx, 'wma_best_w2'] = best_weights[1]
    df_all.at[idx, 'wma_best_w3'] = best_weights[2]
    df_all.at[idx, 'wma_result'] = best_forecast
    df_all.at[idx, 'wma_forecast'] = best_full_forecast  # Store full WMA forecast
    
print(df_all)


            brc      agc            p/n  \
0            20        6       47364317   
1            20        6       47486870   
2            20        6       48017998   
3            20        6       48167835   
4            20        6       48174869   
...         ...      ...            ...   
87146  National  ALL AGC   VI8973679180   
87147  National  ALL AGC  VV12961252100   
87148  National  ALL AGC       X1139855   
87149  National  ALL AGC      XF0202MDN   
87150  National  ALL AGC       Z2230493   

                                                      d   mean_12    std_12  \
0      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
1      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
2      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
3      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
4      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
...      

In [8]:
alpha_ewma = 0.4
def custom_exponential_weighted_moving_average(values, alpha=alpha_ewma):
    ewma_values = [values[0]]  # Start with the first value

    # Apply EWMA formula up to D-2 (i.e., index 11 if length = 12)
    for t in range(1, len(values)):
        if np.isnan(values[t]):
            ewma_t = alpha * 0 + (1 - alpha) * ewma_values[-1]
        else:
            ewma_t = alpha * values[t] + (1 - alpha) * ewma_values[-1]
        ewma_values.append(ewma_t)

    return ewma_values  # This gives you EWMA from D-13 to D-2


def ewma_forecast(data, alpha=alpha_ewma):
    # Calculate EWMA up to D-2
    ewma_up_to_d2 = custom_exponential_weighted_moving_average(data, alpha)

    # Forecast D-1 as same as EWMA at D-2
    ewma_d1 = ewma_up_to_d2[-1]

    # Append D-1 forecast to the EWMA list
    ewma_with_d1 = ewma_up_to_d2 + [ewma_d1]

    # Return full EWMA list (D-13 to D-1) and D-1 forecast
    return ewma_with_d1, ewma_d1

df_all['ewma'], df_all['ewma_result'] = zip(*df_all['clipped_d'].apply(lambda x: ewma_forecast(x[-12:], alpha_ewma)))
print(df_all)


            brc      agc            p/n  \
0            20        6       47364317   
1            20        6       47486870   
2            20        6       48017998   
3            20        6       48167835   
4            20        6       48174869   
...         ...      ...            ...   
87146  National  ALL AGC   VI8973679180   
87147  National  ALL AGC  VV12961252100   
87148  National  ALL AGC       X1139855   
87149  National  ALL AGC      XF0202MDN   
87150  National  ALL AGC       Z2230493   

                                                      d   mean_12    std_12  \
0      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
1      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
2      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
3      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
4      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
...      

In [9]:
logging.info("BEGIN Linear Reggression Calculation")

#LINEAR REGRESSION
#  Calculate Linear Regression
def lr(x):
    df_all = pd.DataFrame()
    df_all['y'] = x
    df_all['x'] = range(1, len(df_all) + 1)
    model =  LinearRegression()
    model.fit(df_all[['x']], df_all['y'])
    df_all.loc[len(df_all), 'x'] = len(df_all) + 1
    return model.predict(df_all[['x']])

df_all['lr'] = df_all['clipped_d'].apply(lambda x: lr(x).tolist())
df_all['lr_result'] = df_all['lr'].apply(lambda x: x[-1:])
print(df_all)

            brc      agc            p/n  \
0            20        6       47364317   
1            20        6       47486870   
2            20        6       48017998   
3            20        6       48167835   
4            20        6       48174869   
...         ...      ...            ...   
87146  National  ALL AGC   VI8973679180   
87147  National  ALL AGC  VV12961252100   
87148  National  ALL AGC       X1139855   
87149  National  ALL AGC      XF0202MDN   
87150  National  ALL AGC       Z2230493   

                                                      d   mean_12    std_12  \
0      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
1      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
2      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
3      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
4      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
...      

In [10]:
logging.info("BEGIN Polynomial Reggression Calculation")

#POLYNOMIAL 2ND AND 3RD
# Calculate Polynomial Regression
def pr(x, pr_degree):
    df_all = pd.DataFrame()
    df_all['y'] = x
    df_all['x'] = range(1, len(df_all) + 1)

    X = df_all[['x']]  # Independent variable (reshape to 2D array)
    y = df_all['y']    # Dependent variable

    poly = PolynomialFeatures(degree=pr_degree)  # Create polynomial features
    X_poly = poly.fit_transform(X)  # Transform input features
    poly_model = LinearRegression()  # Initialize linear regression model
    poly_model.fit(X_poly, y)  # Fit polynomial model

    df_all.loc[len(df_all), 'x'] = len(df_all) + 1
    X_all_poly = poly.transform(df_all[['x']])
    return poly_model.predict(X_all_poly)  

df_all['pr2'] = df_all['clipped_d'].apply(lambda x: pr(x, 2).tolist())
df_all['pr2_result'] = df_all['pr2'].apply(lambda x: x[-1:])
df_all['pr3'] = df_all['clipped_d'].apply(lambda x: pr(x, 3).tolist())
df_all['pr3_result'] = df_all['pr3'].apply(lambda x: x[-1:])
print(df_all)

            brc      agc            p/n  \
0            20        6       47364317   
1            20        6       47486870   
2            20        6       48017998   
3            20        6       48167835   
4            20        6       48174869   
...         ...      ...            ...   
87146  National  ALL AGC   VI8973679180   
87147  National  ALL AGC  VV12961252100   
87148  National  ALL AGC       X1139855   
87149  National  ALL AGC      XF0202MDN   
87150  National  ALL AGC       Z2230493   

                                                      d   mean_12    std_12  \
0      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
1      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
2      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
3      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
4      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
...      

In [11]:
logging.info("BEGIN Simple Exponential Smoothing Calculation")

alpha_ses = 0.8  # ubah nilai alpha (semakin besar semakin berat ke data terbaru)

#SES
def ses(x, alpha = alpha_ses):
    df_all = pd.DataFrame()
    df_all['y'] = x
    df_all['x'] = range(1, len(df_all) + 1)
    df_all.loc[len(df_all), 'x'] = len(df_all) + 1

    new_data = SimpleExpSmoothing(df_all['y']).fit(smoothing_level=alpha, optimized=False).fittedvalues
    return new_data.tolist()

df_all['ses'] = df_all['clipped_d'].apply(lambda x: ses(x, alpha_ses))
df_all['ses_result'] = df_all['ses'].apply(lambda x: x[-1:])
print(df_all)


            brc      agc            p/n  \
0            20        6       47364317   
1            20        6       47486870   
2            20        6       48017998   
3            20        6       48167835   
4            20        6       48174869   
...         ...      ...            ...   
87146  National  ALL AGC   VI8973679180   
87147  National  ALL AGC  VV12961252100   
87148  National  ALL AGC       X1139855   
87149  National  ALL AGC      XF0202MDN   
87150  National  ALL AGC       Z2230493   

                                                      d   mean_12    std_12  \
0      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
1      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
2      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
3      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
4      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
...      

In [12]:
logging.info("BEGIN Double Exponential Smoothing Calculation")

ALPHA = 0.1
BETA = 0.1

def des(x, alpha=ALPHA, beta=BETA):
    df_tmp = pd.DataFrame()
    df_tmp['y'] = x
    df_tmp['x'] = range(1, len(df_tmp) + 1)
    df_tmp.loc[len(df_tmp), 'x'] = len(df_tmp) + 1

    model = ExponentialSmoothing(
        df_tmp['y'],
        trend='add',
        seasonal=None
    )

    fitted_model = model.fit(
        smoothing_level=alpha,
        smoothing_trend=beta,
        optimized=False
    )

    return fitted_model.fittedvalues.tolist()
df_all['des'] = df_all['clipped_d'].apply(lambda x: des(x))
df_all['des_result'] = df_all['des'].apply(lambda x: x[-1])
print(df_all)

            brc      agc            p/n  \
0            20        6       47364317   
1            20        6       47486870   
2            20        6       48017998   
3            20        6       48167835   
4            20        6       48174869   
...         ...      ...            ...   
87146  National  ALL AGC   VI8973679180   
87147  National  ALL AGC  VV12961252100   
87148  National  ALL AGC       X1139855   
87149  National  ALL AGC      XF0202MDN   
87150  National  ALL AGC       Z2230493   

                                                      d   mean_12    std_12  \
0      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
1      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
2      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
3      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
4      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
...      

In [13]:
# Calculate metrics including MASE, MAPE, and SMAPE
def metric(x):
    period_length = len(x['clipped_d'])
    df_all = pd.DataFrame()
    df_all['qty'] = x['clipped_d'][:period_length]  # Ground truth values
    
    # Naive forecast (previous period's value)
    df_all['naive'] = df_all['qty'].shift(1)

    models = ['ma', 'wma_forecast', 'ewma', 'lr', 'pr2', 'pr3', 'ses', 'des']
    for model in models:
        df_all[model] = x[model][:period_length]

    # Compute MASE scaling factor (denominator)
    naive_diff = np.abs(df_all['qty'].diff()).dropna()
    naive_mae = naive_diff.mean() if not naive_diff.empty else np.nan

    result = []
    for model in models:
        y_true = df_all['qty'].dropna()
        y_pred = df_all[model].dropna()
        y_naive = df_all['naive'].dropna()

        # Standard error metrics
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)

        # Relative errors for MdRAE and GMRAE
        relative_errors = np.abs(y_true - y_pred) / np.abs(y_true - y_naive)
        relative_errors = relative_errors.replace([np.inf, -np.inf], np.nan).dropna()

        # Compute MdRAE and GMRAE
        if not relative_errors.empty:
            mdrae = np.median(relative_errors)
            gmrae = np.exp(np.mean(np.log(relative_errors)))
        else:
            mdrae, gmrae = np.nan, np.nan

        # Compute MASE
        mase = mae / naive_mae if naive_mae > 0 else np.nan

        # Compute MAPE (bounded between 0% - 100%)
        mape_values = np.abs((y_true - y_pred) / y_true)
        mape_values = mape_values.replace([np.inf, -np.inf], np.nan).dropna()
        mape = 100 * mape_values.mean() if not mape_values.empty else np.nan

        # Compute SMAPE (bounded between 0% - 100%)
        smape_values = np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-10)  # Avoid div by zero
        smape_values = smape_values.replace([np.inf, -np.inf], np.nan).dropna()
        smape = 100 * smape_values.mean() if not smape_values.empty else np.nan

        result.append({
            'model': model, 'RMSE': rmse, 'MAE': mae, 'R2': r2,
            'MdRAE': mdrae, 'GMRAE': gmrae, 'MASE': mase, 'MAPE': mape, 'SMAPE': smape
        })

    metrics_df_all = pd.DataFrame(result)

    # Select the best model based on MAE
    best_model_row = metrics_df_all.loc[metrics_df_all['MAE'].idxmin()]
    best_model = best_model_row['model']

    return {'best_model': best_model, 'metrics': metrics_df_all.to_dict(orient='records')}

# Apply metric function
df_all['metric'] = df_all.apply(lambda x: metric(x), axis=1)

# Extract best model and metrics
df_all['best_model'] = df_all['metric'].apply(lambda x: x['best_model'])
df_all['metrics'] = df_all['metric'].apply(lambda x: x['metrics'])
df_all = df_all.drop(columns=['metric'])
# Define the number of months
num_months = 13

# Create new columns dynamically for each month
for i in range(num_months, 0, -1):
    df_all[f'pred_{i}'] = df_all.apply(
        lambda x: x[x['best_model']][num_months - i] if pd.notna(x['best_model']) else np.nan, axis=1
    )

# Extract R² of the best model into a new column
def get_best_model_r2(row):
    best_model = row['best_model']
    for m in row['metrics']:
        if m['model'] == best_model:
            return m['R2']
    return np.nan

df_all['best_r2'] = df_all.apply(get_best_model_r2, axis=1)
# Mark R2 performance
df_all['note'] = np.where(df_all['best_r2'] < 0.25, "R2 < 0.25", "Good")
print(df_all)



c:\Users\Brandon\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Brandon\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Brandon\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Brandon\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Brandon\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = ge

            brc      agc            p/n  \
0            20        6       47364317   
1            20        6       47486870   
2            20        6       48017998   
3            20        6       48167835   
4            20        6       48174869   
...         ...      ...            ...   
87146  National  ALL AGC   VI8973679180   
87147  National  ALL AGC  VV12961252100   
87148  National  ALL AGC       X1139855   
87149  National  ALL AGC      XF0202MDN   
87150  National  ALL AGC       Z2230493   

                                                      d   mean_12    std_12  \
0      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
1      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
2      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
3      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
4      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
...      

In [14]:
#kalkulasi semua model D-0
logging.info("BEGIN Data Selection Calculation")
# Select the best model for each row
df_all['mean_12_FD'] = df_all['d'].apply(lambda x: np.mean(x[-12:]))
df_all['std_12_FD'] = df_all['d'].apply(lambda x: np.std(x[-12:]))
df_all['ub_FD'] = df_all['mean_12_FD'] + 1.5 * df_all['std_12_FD']
df_all['clipped_d_FD'] = df_all.apply(lambda row: np.clip(row['d'][-12:], 0, row['ub_FD']).tolist(), axis=1)

In [15]:
logging.info("BEGIN Forecast Alert Scoring")

# =========================
# BASIC STATISTICS
# =========================

df_all['mean_alert'] = df_all['d'].apply(lambda x: np.mean(x[-12:]))
df_all['median_alert'] = df_all['d'].apply(lambda x: np.median(x[-12:]))
df_all['std_alert'] = df_all['d'].apply(lambda x: np.std(x[-12:]))
df_all['max_alert'] = df_all['d'].apply(lambda x: np.max(x[-12:]))

# =========================
# HYBRID SPIKE DETECTION
# =========================

k = 1.8

df_all['spike_alert'] = df_all.apply(
    lambda row:
    1 if row['max_alert'] >
    max(
        row['mean_alert'] * k,
        row['median_alert'] * k
    )
    else 0,
    axis=1
)

# =========================
# COEFFICIENT OF VARIATION
# =========================

df_all['cv_alert'] = df_all.apply(
    lambda row:
    row['std_alert'] / row['mean_alert']
    if row['mean_alert'] != 0 else 0,
    axis=1
)

df_all['volatility_alert'] = df_all['cv_alert'].apply(
    lambda x: 1 if x > 1 else 0
)

# =========================
# INTERMITTENT DEMAND
# =========================

df_all['zero_count_alert'] = df_all['d'].apply(
    lambda x: np.sum(np.array(x[-12:]) == 0)
)

df_all['intermittent_alert'] = df_all['zero_count_alert'].apply(
    lambda x: 1 if x >= 6 else 0
)

# =========================
# FINAL ALERT SCORE
# =========================

df_all['forecast_alert_score'] = (
    df_all['spike_alert']
    + df_all['volatility_alert']
    + df_all['intermittent_alert']
)

# =========================
# FINAL ALERT LABEL
# =========================

df_all['forecast_alert_label'] = df_all['forecast_alert_score'].apply(
    lambda x:
    'HIGH RISK' if x >= 2
    else ('REVIEW' if x == 1 else 'OK')
)

In [16]:
logging.info("BEGIN Moving Average Calculation")

# Calculate Simple Moving Average
df_all['clipped_d_15_FD'] = df_all.apply(lambda row: np.clip(row['d'][-15:], 0, row['ub_FD']).tolist(), axis=1)

# Function to compute SMA forecasts for D-13 to D-1 using 3-point averages
def sma_forecast(data):
    sma_values = []
    for i in range(13):  # We want 13 forecast points: D-13 to D-1
        window = data[i:i+3]
        forecast = np.mean(window)  # Equal weights
        sma_values.append(forecast)
    return sma_values

# Apply SMA forecasting logic
df_all['ma_FD'] = df_all['clipped_d_15_FD'].apply(sma_forecast)

# Extract the last forecast (for D-1)
df_all['ma_result_FD'] = df_all['ma_FD'].apply(lambda x: x[-1])

print(df_all)

            brc      agc            p/n  \
0            20        6       47364317   
1            20        6       47486870   
2            20        6       48017998   
3            20        6       48167835   
4            20        6       48174869   
...         ...      ...            ...   
87146  National  ALL AGC   VI8973679180   
87147  National  ALL AGC  VV12961252100   
87148  National  ALL AGC       X1139855   
87149  National  ALL AGC      XF0202MDN   
87150  National  ALL AGC       Z2230493   

                                                      d   mean_12    std_12  \
0      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
1      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
2      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
3      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
4      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
...      

In [17]:
import numpy as np
import pandas as pd

logging.info("BEGIN Weighted Moving Average Calculation for FD")

# Function to compute WMA forecasts for D-13 to D-1
def wma_forecast_with_weights_FD(data, weights):
    wma_values_FD = []
    for i in range(13):  # Forecasting D-13 to D-1 using D-16 to D-2
        window_FD = data[i:i+3]
        forecast_FD = np.sum(np.array(window_FD) * weights) / sum(weights)
        wma_values_FD.append(forecast_FD)
    return wma_values_FD

# Define step size
step_FD = 0.05

# Initialize columns to store best weights, forecasts, and WMA results
df_all['wma_best_w1_FD'] = np.nan
df_all['wma_best_w2_FD'] = np.nan
df_all['wma_best_w3_FD'] = np.nan
df_all['wma_result_FD'] = np.nan
df_all['wma_forecast_FD'] = df_all.apply(lambda _: [], axis=1)  # Initialize as empty lists

# Optimize weights for each row
for idx, row in df_all.iterrows():
    best_rmse_FD = float('inf')
    best_weights_FD = (0.15, 0.25, 0.6)  # Initial weight assumption
    best_forecast_FD = None
    best_full_forecast_FD = None  # Store full forecast array

    # Iterate over valid w1_FD values
    for w1_FD in np.round(np.arange(0.15, 0.81, step_FD), 2):  # w1_FD ≥ 0.15
        for w2_FD in np.round(np.arange(0.25, 0.86 - w1_FD, step_FD), 2):  # w2_FD ≥ 0.25 and w1_FD + w2_FD ≤ 0.85
            w3_FD = 1 - (w1_FD + w2_FD)  # Ensure sum is exactly 1

            # Ensure w3_FD > w2_FD > w1_FD
            if w3_FD > w2_FD > w1_FD:
                weights_FD = (w1_FD, w2_FD, w3_FD)

                # Compute WMA forecast for this row
                wma_forecast_FD = wma_forecast_with_weights_FD(row['clipped_d_15_FD'], weights_FD)

                # Extract the D-1 prediction (last forecast)
                wma_result_FD = wma_forecast_FD[-1]

                # Extract actual last value of 'd' (D-1)
                d_last_FD = row['d'][-1]

                # Compute RMSE for this row
                rmse_FD = np.sqrt((d_last_FD - wma_result_FD) ** 2)

                # Store best weights if RMSE improves
                if rmse_FD < best_rmse_FD:
                    best_rmse_FD = rmse_FD
                    best_weights_FD = weights_FD
                    best_forecast_FD = wma_result_FD
                    best_full_forecast_FD = wma_forecast_FD  # Store full forecast

    # Store the best weights and forecast for this row
    df_all.at[idx, 'wma_best_w1_FD'] = best_weights_FD[0]
    df_all.at[idx, 'wma_best_w2_FD'] = best_weights_FD[1]
    df_all.at[idx, 'wma_best_w3_FD'] = best_weights_FD[2]
    df_all.at[idx, 'wma_result_FD'] = best_forecast_FD
    df_all.at[idx, 'wma_forecast_FD'] = best_full_forecast_FD  # Store full WMA forecast
print(df_all)

            brc      agc            p/n  \
0            20        6       47364317   
1            20        6       47486870   
2            20        6       48017998   
3            20        6       48167835   
4            20        6       48174869   
...         ...      ...            ...   
87146  National  ALL AGC   VI8973679180   
87147  National  ALL AGC  VV12961252100   
87148  National  ALL AGC       X1139855   
87149  National  ALL AGC      XF0202MDN   
87150  National  ALL AGC       Z2230493   

                                                      d   mean_12    std_12  \
0      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
1      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
2      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
3      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
4      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
...      

In [18]:
# EWMA
alpha_ewma = 0.4

# Custom Exponential Weighted Moving Average Function
def custom_exponential_weighted_moving_average(values, alpha=alpha_ewma):
    ewma_values = [values[0]]  # Start with the first value (D-12)

    # Apply the EWMA formula for D-11 to D-1 (i.e., 11 more steps)
    for t in range(1, len(values)):  # len(values) = 12
        if np.isnan(values[t]):
            ewma_t = alpha * 0 + (1 - alpha) * ewma_values[-1]
        else:
            ewma_t = alpha * values[t] + (1 - alpha) * ewma_values[-1]
        ewma_values.append(ewma_t)
    
    return ewma_values  # EWMA from D-12 to D-1

# Forecast Function Using the Custom EWMA
def ewma_forecast(data, alpha=alpha_ewma):
    # Compute EWMA values for D-12 to D-1
    ewma_values = custom_exponential_weighted_moving_average(data, alpha)

    # Forecast D-0 as the same as EWMA at D-1
    forecast_d0 = ewma_values[-1]

    # Full series includes D-12 to D-0 (13 values total)
    ewma_full = ewma_values + [forecast_d0]

    return ewma_full, forecast_d0

# Apply the EWMA forecast to the dataset
df_all['ewma_FD'], df_all['ewma_result_FD'] = zip(*df_all['clipped_d_FD'].apply(lambda x: ewma_forecast(x[-12:], alpha_ewma)))

print(df_all)


            brc      agc            p/n  \
0            20        6       47364317   
1            20        6       47486870   
2            20        6       48017998   
3            20        6       48167835   
4            20        6       48174869   
...         ...      ...            ...   
87146  National  ALL AGC   VI8973679180   
87147  National  ALL AGC  VV12961252100   
87148  National  ALL AGC       X1139855   
87149  National  ALL AGC      XF0202MDN   
87150  National  ALL AGC       Z2230493   

                                                      d   mean_12    std_12  \
0      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
1      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
2      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
3      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
4      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
...      

In [19]:
#LR
def lr(x):
    df_all = pd.DataFrame()
    df_all['y'] = x
    df_all['x'] = range(1, len(df_all) + 1)
    model =  LinearRegression()
    model.fit(df_all[['x']], df_all['y'])
    df_all.loc[len(df_all), 'x'] = len(df_all) + 1
    return model.predict(df_all[['x']])
df_all['lr_FD'] = df_all['clipped_d_FD'].apply(lambda x: lr(x).tolist())
df_all['lr_result_FD'] = df_all['lr_FD'].apply(lambda x: x[-1:])
print(df_all)

            brc      agc            p/n  \
0            20        6       47364317   
1            20        6       47486870   
2            20        6       48017998   
3            20        6       48167835   
4            20        6       48174869   
...         ...      ...            ...   
87146  National  ALL AGC   VI8973679180   
87147  National  ALL AGC  VV12961252100   
87148  National  ALL AGC       X1139855   
87149  National  ALL AGC      XF0202MDN   
87150  National  ALL AGC       Z2230493   

                                                      d   mean_12    std_12  \
0      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
1      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
2      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
3      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
4      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
...      

In [20]:
#PR2&3
def pr(x, pr_degree):
    df_all = pd.DataFrame()
    df_all['y'] = x
    df_all['x'] = range(1, len(df_all) + 1)
    X = df_all[['x']]  # Independent variable (reshape to 2D array)
    y = df_all['y']    # Dependent variable
    poly = PolynomialFeatures(degree=pr_degree)  # Create polynomial features
    X_poly = poly.fit_transform(X)  # Transform input features
    poly_model = LinearRegression()  # Initialize linear regression model
    poly_model.fit(X_poly, y)  # Fit polynomial model
    df_all.loc[len(df_all), 'x'] = len(df_all) + 1
    X_all_poly = poly.transform(df_all[['x']])
    return poly_model.predict(X_all_poly)  
df_all['pr2_FD'] = df_all['clipped_d_FD'].apply(lambda x: pr(x, 2).tolist())
df_all['pr2_result_FD'] = df_all['pr2_FD'].apply(lambda x: x[-1:])
df_all['pr3_FD'] = df_all['clipped_d_FD'].apply(lambda x: pr(x, 3).tolist())
df_all['pr3_result_FD'] = df_all['pr3_FD'].apply(lambda x: x[-1:])
display(df_all)

,brc,agc,p/n,d,mean_12,std_12,ub,clipped_d,clipped_d_15,ma,...,wma_result_FD,wma_forecast_FD,ewma_FD,ewma_result_FD,lr_FD,lr_result_FD,pr2_FD,pr2_result_FD,pr3_FD,pr3_result_FD
0,20,6,47364317,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.00000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0]
1,20,6,47486870,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.00000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0]
2,20,6,48017998,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.00000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0]
3,20,6,48167835,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.00000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0]
4,20,6,48174869,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.00000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87146,National,ALL AGC,VI8973679180,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.00000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0]
87147,National,ALL AGC,VV12961252100,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.00000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0

In [21]:
#SES
def ses(x, alpha = alpha_ses):
    df_all = pd.DataFrame()
    df_all['y'] = x
    df_all['x'] = range(1, len(df_all) + 1)
    df_all.loc[len(df_all), 'x'] = len(df_all) + 1
    new_data = SimpleExpSmoothing(df_all['y']).fit(smoothing_level=alpha, optimized=False).fittedvalues
    return new_data.tolist()
df_all['ses_FD'] = df_all['clipped_d_FD'].apply(lambda x: ses(x, alpha_ses))
df_all['ses_result_FD'] = df_all['ses_FD'].apply(lambda x: x[-1:])
print(df_all)

            brc      agc            p/n  \
0            20        6       47364317   
1            20        6       47486870   
2            20        6       48017998   
3            20        6       48167835   
4            20        6       48174869   
...         ...      ...            ...   
87146  National  ALL AGC   VI8973679180   
87147  National  ALL AGC  VV12961252100   
87148  National  ALL AGC       X1139855   
87149  National  ALL AGC      XF0202MDN   
87150  National  ALL AGC       Z2230493   

                                                      d   mean_12    std_12  \
0      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
1      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
2      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
3      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
4      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
...      

In [22]:
# DES - FD
logging.info("BEGIN Double Exponential Smoothing FD Calculation")

ALPHA_FD = 0.1
BETA_FD = 0.1

# Double Exponential Smoothing function for FD
def des_FD(x, alpha=ALPHA_FD, beta=BETA_FD):
    df_tmp = pd.DataFrame()
    df_tmp['y'] = x
    df_tmp['x'] = range(1, len(df_tmp) + 1)
    df_tmp.loc[len(df_tmp), 'x'] = len(df_tmp) + 1

    model = ExponentialSmoothing(
        df_tmp['y'],
        trend='add',
        seasonal=None
    )

    fitted_model = model.fit(
        smoothing_level=alpha,
        smoothing_trend=beta,
        optimized=False
    )

    return fitted_model.fittedvalues.tolist()
df_all['des_FD'] = df_all['clipped_d_FD'].apply(lambda x: des_FD(x))
df_all['des_result_FD'] = df_all['des_FD'].apply(lambda x: x[-1])


In [23]:
logging.info("BEGIN Metric Calculation for _FD")

# Calculate metrics including MdRAE, GMRAE, MASE, MAPE, and SMAPE
def metric_FD(x):
    period_length = len(x['clipped_d_FD'])
    df_all = pd.DataFrame()
    df_all['qty'] = x['clipped_d_FD'][:period_length]  # Ground truth values

    # Naive forecast (previous period's value)
    df_all['naive'] = df_all['qty'].shift(1)

    models = ['ma_FD', 'wma_forecast_FD', 'ewma_FD', 'lr_FD', 'pr2_FD', 'pr3_FD', 'ses_FD', 'des_FD']
    for model in models:
        df_all[model] = x[model][:period_length]

    # Compute MASE scaling factor (denominator)
    naive_diff = np.abs(df_all['qty'].diff()).dropna()
    naive_mae = naive_diff.mean() if not naive_diff.empty else np.nan

    result = []
    for model in models:
        y_true = df_all['qty'].dropna()
        y_pred = df_all[model].dropna()
        y_naive = df_all['naive'].dropna()

        # Standard error metrics
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)

        # Relative errors for MdRAE and GMRAE
        relative_errors = np.abs(y_true - y_pred) / np.abs(y_true - y_naive)
        relative_errors = relative_errors.replace([np.inf, -np.inf], np.nan).dropna()

        # Compute MdRAE and GMRAE
        if not relative_errors.empty:
            mdrae = np.median(relative_errors)
            gmrae = np.exp(np.mean(np.log(relative_errors)))
        else:
            mdrae, gmrae = np.nan, np.nan

        # Compute MASE
        mase = mae / naive_mae if naive_mae > 0 else np.nan

        # Compute MAPE (bounded between 0% - 100%)
        mape_values = np.abs((y_true - y_pred) / y_true)
        mape_values = mape_values.replace([np.inf, -np.inf], np.nan).dropna()
        mape = 100 * mape_values.mean() if not mape_values.empty else np.nan

        # Compute SMAPE (bounded between 0% - 100%)
        smape_values = np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-10)  # Avoid div by zero
        smape_values = smape_values.replace([np.inf, -np.inf], np.nan).dropna()
        smape = 100 * smape_values.mean() if not smape_values.empty else np.nan

        result.append({
            'model': model, 'RMSE': rmse, 'MAE': mae, 'R2': r2,
            'MdRAE': mdrae, 'GMRAE': gmrae, 'MASE': mase, 'MAPE': mape, 'SMAPE': smape
        })

    return result  # Returning the metrics list

# Apply the metric function
df_all['metrics_FD'] = df_all.apply(lambda x: metric_FD(x), axis=1)

def get_best_r2_FD(row):
    best_model = row['best_model']
    metrics_fd = row.get('metrics_FD', [])
    for m in metrics_fd:
        if m['model'] == best_model + '_FD':
            return m['R2']
    return np.nan
df_all['best_r2_FD'] = df_all.apply(get_best_r2_FD, axis=1)
df_all['r2_status_FD'] = np.where(df_all['best_r2_FD'] < 0.25, "R2 < 0.25", "Good")

print(df_all)   

c:\Users\Brandon\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Brandon\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Brandon\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Brandon\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Brandon\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = ge

            brc      agc            p/n  \
0            20        6       47364317   
1            20        6       47486870   
2            20        6       48017998   
3            20        6       48167835   
4            20        6       48174869   
...         ...      ...            ...   
87146  National  ALL AGC   VI8973679180   
87147  National  ALL AGC  VV12961252100   
87148  National  ALL AGC       X1139855   
87149  National  ALL AGC      XF0202MDN   
87150  National  ALL AGC       Z2230493   

                                                      d   mean_12    std_12  \
0      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
1      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
2      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
3      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
4      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
...      

In [ ]:
def apply_best_model_forecast(row):
    best_model = row['best_model']
    if best_model == 'ma':
        return row['ma_result_FD']
    elif best_model == 'wma':
        return row['wma_result_FD']
    elif best_model == 'ewma':
        return row['ewma_result_FD']
    elif best_model == 'lr':
        return row['lr_result_FD'][-1] if isinstance(row['lr_result_FD'], list) else row['lr_result_FD']
    elif best_model == 'pr2':
        return row['pr2_result_FD'][-1] if isinstance(row['pr2_result_FD'], list) else row['pr2_result_FD']
    elif best_model == 'pr3':
        return row['pr3_result_FD'][-1] if isinstance(row['pr3_result_FD'], list) else row['pr3_result_FD']
    elif best_model == 'ses':
        return row['ses_result_FD'][-1] if isinstance(row['ses_result_FD'], list) else row['ses_result_FD']
    elif best_model == 'des':
        return row['des_result_FD'][-1] if isinstance(row['des_result_FD'], list) else row['des_result_FD']
    else:
        return np.nan
    
df_all['FD_forecast'] = df_all.apply(apply_best_model_forecast, axis=1)
# Define the number of months (from 12 to 1, excluding 0)
num_months = 13  # Total months (D-12 to D-0), but we exclude D-0

# Map best model to the correct forecast series (excluding pred_0_FD)
def extract_forecast_values(row, month_idx):
    best_model = row['best_model']
    forecast_column = f"{best_model}_FD"  # Example: 'ma_result_FD', 'wma_result_FD'
    
    if forecast_column in row and isinstance(row[forecast_column], list):
        if len(row[forecast_column]) >= (13 - month_idx):
            return row[forecast_column][12 - month_idx]  # Extract the correct past forecast
    return np.nan  # Return NaN if data is missing or not a list

# Create columns for pred_12_FD to pred_1_FD
for i in range(num_months - 1, 0, -1):  # From 12 to 1
    df_all[f'pred_{i}_FD'] = df_all.apply(lambda x: extract_forecast_values(x, i), axis=1)

# Ensure FD_forecast contains only numeric values
df_all['FD_final'] = np.maximum(0, df_all['FD_forecast'].round().astype(int))
# Get all columns except the last four we want to reorder
columns_to_keep = [col for col in df_all.columns if col not in ['best_model', 'metrics', 'FD_forecast', 'FD_final']]

# Define the new order with the last four columns at the end
column_order = columns_to_keep + ['best_model', 'metrics', 'FD_forecast', 'FD_final']

# Reorder DataFrame
df_all = df_all[column_order]

print(df_all)


            brc      agc            p/n  \
0            20        6       47364317   
1            20        6       47486870   
2            20        6       48017998   
3            20        6       48167835   
4            20        6       48174869   
...         ...      ...            ...   
87146  National  ALL AGC   VI8973679180   
87147  National  ALL AGC  VV12961252100   
87148  National  ALL AGC       X1139855   
87149  National  ALL AGC      XF0202MDN   
87150  National  ALL AGC       Z2230493   

                                                      d   mean_12    std_12  \
0      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
1      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
2      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
3      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
4      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  0.000000  0.000000   
...      

In [25]:
logging.info("Forecast Calculation Completed")

In [26]:
df_final = df_all[
    [
        # =========================
        # IDENTIFICATION
        # =========================
        'brc',
        'agc',
        'p/n',

        # =========================
        # ALL FORECAST MODELS
        # =========================
        'clipped_d_FD',
        'ma_FD',
        'wma_forecast_FD',
        'ewma_FD',
        'lr_FD',
        'pr2_FD',
        'pr3_FD',
        'ses_FD',
        'des_FD',

        # =========================
        # MODEL SELECTION
        # =========================
        'best_model',
        'FD_forecast',
        'FD_final',
        
        # =========================
        # INDICATORS
        # =========================        
        'metrics_FD',
        'best_r2_FD',
        'r2_status_FD',

        # =========================
        # ALERT DETAILS
        # =========================
        'spike_alert',
        'volatility_alert',
        'intermittent_alert',

        # =========================
        # ALERT SYSTEM
        # =========================
        'forecast_alert_score',
        'forecast_alert_label'        
    ]
].copy()

In [27]:
logging.info("Begin Creating Excel For DataFrame")

# if output folder not exist, create it
if not os.path.exists("output"):
    os.makedirs("output")

# Create Excel File, filename with date
filename = "output/forecast_" + time.strftime("%Y-%m-%d") + ".xlsx"

# Save DataFrame to Excel
df_final.to_excel(filename, index=False)

# Get the file size in MB
file_size = os.path.getsize(filename) / (1024 * 1024)

logging.info(f"Excel File Created: {filename}, Size: {file_size:.2f} MB")

